# Final analysis

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from sklearn.cluster import AgglomerativeClustering

from social_graph.describe import persona_significance
from social_graph.pipeline import run_simulation_preprocessing
from social_graph.preprocessing import *
from social_graph.metrics import cramers_v_matrix

In [ ]:
conn = sqlite3.connect('data/exp_set_2/database_server.db')

In [ ]:
users = pd.read_sql('SELECT * FROM user_mgmt', conn)
users.head()

## Persona creation - comparison KMeans vs. Hierarchical clustering

In [ ]:
features_cols = ['openness', 'conscientiousness', 'extroversion', 'agreeableness', 'neuroticism']
personas, features_df, follow = run_simulation_preprocessing(conn, feature_cols=features_cols)

In [ ]:
scores = []

for k in range(2, 10):
    hier = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = hier.fit_predict(features_df)
    score = silhouette_score(features_df, labels)
    scores.append(score)
    print(f"k={k} silhouette={score:.4f}")

plt.plot(range(2, 10), scores)
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.show()

In [ ]:
personas = create_personae(6, personas, features_df)

In [ ]:
hier = AgglomerativeClustering(n_clusters=6, linkage='ward')
personas['hier_clusters'] = hier.fit_predict(features_df)
personas['hier_clusters'].value_counts()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(features_df)

In [ ]:
plot_df = pd.DataFrame({
    'PC1': X_pca[:,0],
    'PC2': X_pca[:,1],
    'kmeans': personas['persona'],
    'hierarchical': personas['hier_clusters'],
    'profession': personas['profession']
})

In [ ]:
def plot_pca(plot_df, colour_by=None, title='PCA'):
    # unique interests
    features = plot_df[colour_by].unique()
    # assign colors
    cmap = plt.cm.get_cmap('tab20', len(features))

    plt.figure(figsize=(8, 6))
    for i, colour in enumerate(features):
        subset = plot_df[plot_df[colour_by] == colour]

        plt.scatter(
            subset['PC1'],
            subset['PC2'],
            label=colour,
            alpha=0.7
        )

    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title(title)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

In [ ]:
plot_pca(plot_df, colour_by='kmeans', title='KMeans')

In [ ]:
plot_pca(plot_df, colour_by='hierarchical', title='hierarchical clusters')

In [ ]:
adjusted_rand_score(personas['persona'], personas['hier_clusters'])

In [ ]:
print(silhouette_score(features_df, personas['persona']))
print(silhouette_score(features_df, personas['hier_clusters']))

K-means clustering was selected as the primary persona extraction method because it consistently achieved slightly higher silhouette scores on the synthetic (YSocial) dataset and substantially better scores on the Reddit dataset. Additionally, PCA visualizations of Big Five features showed that K-means clusters were more compact and better separated, suggesting clearer personality differentiation between personas.

In [ ]:
cramers_v_matrix(personas[features_cols + ['persona']], label='Persona')

In [ ]:
cramers_v_matrix(personas[features_cols + ['hier_clusters']], label='Persona')

KMeans is better.

## Big Five vs. Professions

In [ ]:
plot_pca(plot_df, colour_by='profession', title='Professions')

It's impossible to plot it since we have only 32 big five traits combinations:(

The synthetic dataset contained only 32 possible Big Five combinations, limiting personality variance and naturally reducing the separability of profession groups in PCA space.

In [ ]:
personaeb1, features_df_b1, followb1 = run_simulation_preprocessing(conn, label='Simulation_b1', feature_cols=features_cols)

In [ ]:
from social_graph.pipeline import *
k_best = 6

description_b1, persona_dict = cluster_persona_and_analyse(personaeb1, features_df_b1, k_best, label='Simulation_b1', feature_cols=features_cols)

In [ ]:
global_metrics_b1, summary_b1 = build_graph_and_analyse(followb1, persona_dict, label='Simulation_b1')

In [ ]:
summary_b1

In [ ]:
description_b1

In [ ]:
personas.drop(['hier_clusters'], axis=1, inplace=True)
personas.head()

In [ ]:
# personas.to_csv('data/exp_set_2/persona_df.csv')